In [7]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "126ac41d-e5e6-4702-b949-9d141a1b7042"
RESOURCE_GROUP = "rg-mlab1-ali886"
WORKSPACE = "mlw-lab1-ali886"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
                     RESOURCE_GROUP, WORKSPACE)
print("Connected:", ml_client.workspace_name)

Connected: mlw-lab1-ali886


In [10]:
try:
    ml_client.online_endpoints.begin_delete(name="knn-maint-ali886").result()
    print("Old endpoint deleted")
except Exception as e:
    print("Delete message:", e)

....Old endpoint deleted


In [11]:
ENDPOINT_NAME = "knn-maint-ali886"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created:", ENDPOINT_NAME)

Endpoint created: knn-maint-ali886


In [12]:
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

Deploying model: knn-maintenance-notebook version 4
.................................................................................................................................................

HttpResponseError: (BadArgument) User container has crashed or terminated: Liveness probe failed: HTTP probe failed with statuscode: 502. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated: Liveness probe failed: HTTP probe failed with statuscode: 502. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [13]:
import json

sample = {"input_data": [[298.1, 308.6, 1551, 42.8, 0, 0]]}  # apne features ke hisaab se
with open("../deployment/sample.json", "w") as f:
    json.dump(sample, f)

result = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file="../deployment/sample.json",
)
print(result)

MlException: The specified endpoint does not exist, please check if the endpoint name is correct and the endpoint creation was successful.


In [14]:
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print(model.type, model.path)

import sklearn
print(sklearn.__version__)

mlflow_model azureml://subscriptions/126ac41d-e5e6-4702-b949-9d141a1b7042/resourceGroups/rg-mlab1-ali886/workspaces/mlw-lab1-ali886/datastores/workspaceartifactstore/paths/ExperimentRun/dcid.417c9ee8-5b71-480c-bc83-5425abfa87f0/model
1.7.2


In [15]:
import os
from azure.ai.ml.entities import Environment

# 1) conda.yml banao
os.makedirs("../deployment", exist_ok=True)
open("../deployment/conda.yml", "w").write('''
name: knn-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - mlflow
      - cloudpickle
      - scikit-learn==1.7.2
      - numpy
      - pandas
      - azureml-inference-server-http
''')

# 2) environment
env = Environment(
    name="knn-mlflow-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.2-ubuntu20.04:latest",
    conda_file="../deployment/conda.yml",
)

# 3) model
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

# 4) deployment
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    environment=env,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

# 5) traffic
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

Deploying model: knn-maintenance-notebook version 4
.....................

HttpResponseError: (ImageBuildFailure) Deployment failed due to reported user error. UserError: Image mcr.microsoft.com/azureml/openmpi4.1.2-ubuntu20.04:latest for environment knn-mlflow-env does not exist in registry.\\\",\\\"Data\\\":{\\\"ErrorType\\\":\\\"ImageError\\\",\\\"IsTerminal\\\":true,\\\"ODataError\\\":{\\\"code\\\":\\\"UserError\\\",\\\"message\\\":\\\"Image mcr.microsoft.com/azureml/openmpi4.1.2-ubuntu20.04:latest for environment knn-mlflow-env does not exist in registry.\\\",\\\"details\\\":[]},\\\"isUserError\\\":true},\\\"InnerException\\\":null,\\\"HelpURL\\\":null,\\\"StackTraceString\\\":\\\"   at Microsoft.MachineLearning.InferenceDeployment.Silo.Grains.Flows.OnlineImageAndModelPrepFlow.<>c__DisplayClass6_0.<<ExecuteTurnAsync>b__3>d.MoveNext() in /mnt/vss/_work/1/s/src/Silo/Grains/Flows/OnlineImageAndModelPrepFlow.cs:line 252\\\\n--- End of stack trace from previous location ---\\\\n   at Microsoft.MachineLearning.InferenceDeployment.Silo.Grains.Flows.SequentialFl

In [16]:
import os
from azure.ai.ml.entities import Environment

os.makedirs("../deployment", exist_ok=True)
open("../deployment/conda.yml", "w").write('''
name: knn-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - mlflow
      - cloudpickle
      - scikit-learn==1.7.2
      - numpy
      - pandas
      - azureml-inference-server-http
''')

env = Environment(
    name="knn-mlflow-env-v2",
    image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cpu-inference:latest",
    conda_file="../deployment/conda.yml",
)

model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    environment=env,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint knn-maint-ali886 exists


......................................................................................................................................................................................

HttpResponseError: (BadArgument) User container has crashed or terminated: Liveness probe failed: HTTP probe failed with statuscode: 502. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated: Liveness probe failed: HTTP probe failed with statuscode: 502. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [17]:
print(ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=ENDPOINT_NAME, lines=300))

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-08T11:10:04.32115Z, Message: Readiness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-08T11:10:14.320034Z, Message: Readiness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: LivenessProbeFailed, Type: Warning, Time: 2026-10-08T11:10:14.320129Z, Message: Liveness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: LivenessProbeFailed, Type: Warning, Time: 2026-10-08T11:10:24.646835Z, Message: Liveness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-08T11:10:24.646965Z, Message: Readiness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning

In [19]:
import os
from azure.ai.ml.entities import Environment, CodeConfiguration

os.makedirs("../deployment", exist_ok=True)

open("../deployment/score.py", "w").write('''
import os, json
import mlflow.pyfunc
import pandas as pd

def init():
    global model
    base = os.getenv("AZUREML_MODEL_DIR")
    model_path = base
    for root, _, files in os.walk(base):
        if "MLmodel" in files:
            model_path = root
            break
    model = mlflow.pyfunc.load_model(model_path)

def run(raw_data):
    data = json.loads(raw_data)["input_data"]
    if isinstance(data, dict):
        df = pd.DataFrame(data["data"], columns=data.get("columns"))
    else:
        df = pd.DataFrame(data)
    return model.predict(df).tolist()
''')

open("../deployment/conda.yml", "w").write('''
name: knn-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - mlflow
      - cloudpickle
      - scikit-learn==1.7.2
      - numpy
      - pandas
      - azureml-inference-server-http
''')

env = Environment(
    name="knn-mlflow-env-v3",
    image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cpu-inference:latest",
    conda_file="../deployment/conda.yml",
)

model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    environment=env,
    code_configuration=CodeConfiguration(code="../deployment", scoring_script="score.py"),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

Check: endpoint knn-maint-ali886 exists


........Deployment ready


In [21]:
import json

sample = {"input_data": [[298.1, 308.6, 1551, 42.8, 0, 0]]}  # apne 6 features ke order mein
with open("../deployment/sample.json", "w") as f:
    json.dump(sample, f)

result = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file="../deployment/sample.json",
)
print(result)

HttpResponseError: (None) An unexpected error occurred in scoring script. Check the logs for more info.
Code: None
Message: An unexpected error occurred in scoring script. Check the logs for more info.

In [22]:
logs = ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=ENDPOINT_NAME, lines=300)
print(logs[-3500:])

odel.predict(df).tolist()
  File "/azureml-envs/azureml_11fba46604255bca02aa5991396c5783/lib/python3.10/site-packages/mlflow/pyfunc/__init__.py", line 846, in predict
    return self._predict(data, params)
  File "/azureml-envs/azureml_11fba46604255bca02aa5991396c5783/lib/python3.10/site-packages/mlflow/pyfunc/__init__.py", line 885, in _predict
    data, params = _validate_prediction_input(
  File "/azureml-envs/azureml_11fba46604255bca02aa5991396c5783/lib/python3.10/site-packages/mlflow/pyfunc/__init__.py", line 744, in _validate_prediction_input
    raise MlflowException.invalid_parameter_value(
mlflow.exceptions.MlflowException: Failed to enforce schema of data '       0      1     2     3  4  5
0  298.1  308.6  1551  42.8  0  0' with schema '['air_temp_k': double (required), 'process_temp_k': double (required), 'rpm': double (required), 'torque_nm': double (required), 'tool_wear_min': double (required), 'type': string (required)]'. Error: Model is missing inputs ['air_temp_k', 'pr

In [23]:
import json

sample = {
    "input_data": {
        "columns": ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min", "type"],
        "data": [
            [298.1, 308.6, 1551.0, 42.8, 0.0, "M"],     # normal machine
            [300.0, 309.5, 1300.0, 65.0, 220.0, "L"]    # high torque + high wear (failure-like)
        ]
    }
}

with open("../deployment/sample.json", "w") as f:
    json.dump(sample, f)

result = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file="../deployment/sample.json",
)
print(result)

[0, 1]
